[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/08_embedding_visualization.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/08_embedding_visualization.ipynb)

# Visualisierung von Text-Embeddings und Vergleich mit Augmentations-Daten

Dieses Notebook visualisiert die Embedding-Vektoren des 20 Newsgroups Datensatzes in einem 2D-Plot. Wir nutzen ein Embedding-Modell von Hugging Face und reduzieren die Dimensionen mittels PCA (Principal Component Analysis), um auch neue, vom Nutzer eingegebene Texte (z.B. generierte Texte aus der Datenaugmentierung) in denselben Vektorraum projizieren zu können.

Das Ziel ist es zu prüfen, ob ein generierter Satz einen ähnlichen Embedding-Vektor erzeugt wie die Texte der Zielklasse.

In [ ]:
# Installation notwendiger Bibliotheken
!pip install scikit-learn matplotlib numpy llama-index-embeddings-huggingface

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import torch

## 1. Datensatz laden

Wir laden den 20 Newsgroups Datensatz.

In [ ]:
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
class_names = newsgroups.target_names

print(f"Anzahl der Klassen: {len(class_names)}")
print(f"Anzahl der Dokumente: {len(newsgroups.data)}")

## 2. Embedding Modell initialisieren

Wir nutzen das Modell `intfloat/e5-small-v2` von Hugging Face.

In [ ]:
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-small-v2")

def get_embeddings(texts):
    # E5 Modelle benötigen oft ein Präfix wie 'query: ' oder 'passage: '
    prefixed_texts = [f"passage: {t[:512]}" for t in texts]
    embeddings = [embed_model.get_text_embedding(t) for t in prefixed_texts]
    return np.array(embeddings)

## 3. Embeddings generieren und Dimensionen reduzieren

Wir begrenzen die Anzahl der Dokumente für die Visualisierung auf 30 pro Klasse, um die Berechnung zu beschleunigen und den Plot übersichtlich zu halten.

In [ ]:
n_samples_per_class = 30
indices = []
for i in range(len(class_names)):
    class_indices = np.where(newsgroups.target == i)[0]
    selected = np.random.choice(class_indices, min(len(class_indices), n_samples_per_class), replace=False)
    indices.extend(selected)

sample_data = [newsgroups.data[i] for i in indices]
sample_targets = newsgroups.target[indices]

print("Generiere Embeddings...")
embeddings = get_embeddings(sample_data)

# PCA zur Dimensionsreduktion auf 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Klassen-Mittelwerte (Centroids) im originalen Embedding-Raum berechnen
centroids = {}
for i in range(len(class_names)):
    class_emb = embeddings[sample_targets == i]
    centroids[i] = np.mean(class_emb, axis=0)

print("Fertig.")

## 4. Visualisierung

Wir plotten die Embeddings der Trainingsdaten.

In [ ]:
plt.figure(figsize=(15, 10))
plt.rcParams.update({'font.size': 14})

for i, class_name in enumerate(class_names):
    mask = sample_targets == i
    plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], label=class_name, alpha=0.6)

plt.title("2D Visualisierung von Newsgroup Embeddings (PCA)")
plt.xlabel("Hauptkomponente 1")
plt.ylabel("Hauptkomponente 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Eigener Textvergleich

Geben Sie hier einen Text ein (z.B. ein Ergebnis aus der Datenaugmentierung), um ihn im Vektorraum zu sehen.

In [ ]:
#@title Nutzereingabe
user_text = "Religious beliefs and the existence of a higher power are central to human history." #@param {type:"string"}
target_class_for_comparison = "soc.religion.christian" #@param ["alt.atheism", "comp.graphics", "comp.os.ms-windows.misc", "comp.sys.ibm.pc.hardware", "comp.sys.mac.hardware", "comp.windows.x", "misc.forsale", "rec.autos", "rec.motorcycles", "rec.sport.baseball", "rec.sport.hockey", "sci.crypt", "sci.electronics", "sci.med", "sci.space", "soc.religion.christian", "talk.politics.guns", "talk.politics.mideast", "talk.politics.misc", "talk.religion.misc"]

# Embedding für Nutzer-Text
user_embedding = get_embeddings([user_text])[0]

# Projektion in 2D
user_embedding_2d = pca.transform(user_embedding.reshape(1, -1))

# Cosine Similarity zu allen Klassen-Mittelwerten
similarities = {}
for i, class_name in enumerate(class_names):
    sim = cosine_similarity(user_embedding.reshape(1, -1), centroids[i].reshape(1, -1))[0][0]
    similarities[class_name] = sim

print("--- Cosine Similarity zu den Klassen-Mittelwerten ---")
for class_name, sim in similarities.items():
    print(f"{class_name: <25}: {sim:.4f}")

# Visualisierung mit dem neuen Punkt
plt.figure(figsize=(15, 10))
for i, class_name in enumerate(class_names):
    mask = sample_targets == i
    plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], label=class_name, alpha=0.3)

plt.scatter(user_embedding_2d[0, 0], user_embedding_2d[0, 1], color='red', marker='X', s=300, label='Nutzer Text', edgecolors='black')

plt.title(f"Position des Nutzer-Textes im Embedding-Raum\nZielklasse: {target_class_for_comparison}")
plt.xlabel("Hauptkomponente 1")
plt.ylabel("Hauptkomponente 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()